# Image preprocessing — Second-Progression target

Produces the artefacts that Exp(iv) — **Molecular + Treatment +
Timepoints + VLM** — needs at training time.  The actual VLM
inference runs **once** offline; the Mistral pipeline only sees the
resulting captions through `mri_captions.csv`.

## Outputs

| Artefact | Purpose |
|---|---|
| `Processed/mri_paths.csv` | One row per (patient × pre-T₂ × complete) scan; absolute paths to t1c / t1n / t2f / t2w / tumourMask. |
| `Processed/mri_summary.json` | Coverage stats. |
| `../VLM/run_radfm_captions.py` | Caption pipeline — tries RadFM, then LLaVA-Med, then MAIRA-2, then Med-Flamingo. |
| `../VLM/README.md` | Setup + how to run + which model was used. |

The medical-VLM fallback ladder, in priority order:

1. **RadFM** (Wu et al. 2024) — 13B, trained on 16M radiology pairs
   *including 3D brain MRI*; native multi-volume support.
2. **LLaVA-Med v1.5** (Microsoft, 2024) — 7B; PMC-15M biomedical
   pre-training; reliable MLX port.
3. **MAIRA-2** (Microsoft, 2024) — 7B; radiology report generation.
4. **Med-Flamingo** (Stanford, 2023) — 9B; few-shot medical multi-modal.

The captioning script writes `Processed/mri_captions.csv` and
`Processed/mri_captions.meta.json` recording **which model was actually
used** so the proposal write-up always reflects reality.


In [ ]:
import sys, json, re
sys.path.insert(0, ".")
import pandas as pd, numpy as np
from pathlib import Path
from _landmark import (load_raw, eligible_cohort, attach_y_and_landmark, MRI_DAY_COLS)

PROC = Path("Processed"); PROC.mkdir(exist_ok=True)


## 1. Re-inventory the NIfTI tree


In [ ]:
ROOT = Path("Raw/MRI").resolve()
PAT_RE = re.compile(r"(PatientID_\d+)_Timepoint_(\d+)_(brain_t1c|brain_t1n|brain_t2f|brain_t2w|tumorMask)\.nii\.gz$")
records = []
for pdir in sorted(ROOT.iterdir()):
    if not pdir.is_dir(): continue
    for tdir in sorted(pdir.iterdir()):
        if not tdir.is_dir() or not tdir.name.startswith("Timepoint_"): continue
        for f in tdir.iterdir():
            m = PAT_RE.search(f.name)
            if not m: continue
            records.append({"Patient_ID": m.group(1), "Timepoint": int(m.group(2)),
                            "modality": m.group(3), "path": str(f.resolve())})
inv = pd.DataFrame(records)
print(f"  {len(inv)} files across {inv['Patient_ID'].nunique()} patients")


## 2. Apply the eligibility + Landmark_day + Death_day gates

We apply two gates in sequence, logging every dropped scan to
`Processed/mri_drop_log.csv` for the audit:

1. **Landmark gate**: scan must satisfy `Day_from_diag < Landmark_day`
   (don't show the model future scans).
2. **Death gate**: scan must satisfy `Day_from_diag ≤ Death_day`
   (don't show the model post-mortem scans — those are upstream
   data-entry errors).


In [ ]:
raw = load_raw()
elig = attach_y_and_landmark(eligible_cohort(raw))
day_map = raw.set_index("Patient_ID")[MRI_DAY_COLS].copy()
day_map.columns = list(range(1, len(MRI_DAY_COLS)+1))

inv = inv[inv["Patient_ID"].isin(elig["Patient_ID"])].copy()
inv["Day_from_diag"] = [
    float(day_map.at[p, t]) if (p in day_map.index and t in day_map.columns and pd.notna(day_map.at[p, t])) else np.nan
    for p, t in zip(inv["Patient_ID"], inv["Timepoint"])
]
inv = inv.merge(elig.set_index("Patient_ID")[["y","Landmark_day","Death_day"]],
                left_on="Patient_ID", right_index=True, how="left")

drop_log = []

# Gate 1: must be pre-Landmark_day
pre_landmark_mask = inv["Day_from_diag"] < inv["Landmark_day"]
post_lm = inv[~pre_landmark_mask & inv["Day_from_diag"].notna()]
for _, r in post_lm.drop_duplicates(["Patient_ID","Timepoint"]).iterrows():
    drop_log.append({"Patient_ID": r["Patient_ID"], "Timepoint": int(r["Timepoint"]),
                     "Day_from_diag": r["Day_from_diag"], "Landmark_day": r["Landmark_day"],
                     "Death_day": r["Death_day"], "reason": "post-landmark"})
inv = inv[pre_landmark_mask | inv["Day_from_diag"].isna()].copy()

# Gate 2: if Death_day known, must be ≤ Death_day
death_known = inv["Death_day"].notna()
post_death = death_known & (inv["Day_from_diag"] > inv["Death_day"])
for _, r in inv[post_death].drop_duplicates(["Patient_ID","Timepoint"]).iterrows():
    drop_log.append({"Patient_ID": r["Patient_ID"], "Timepoint": int(r["Timepoint"]),
                     "Day_from_diag": r["Day_from_diag"], "Landmark_day": r["Landmark_day"],
                     "Death_day": r["Death_day"], "reason": "post-death"})
inv = inv[~post_death].copy()

# Gate 3: must have a recorded scan day (otherwise we can't gate it at all)
inv = inv[inv["Day_from_diag"].notna()].copy()

drop_df = pd.DataFrame(drop_log).drop_duplicates(["Patient_ID","Timepoint","reason"])
drop_df.to_csv(PROC / "mri_drop_log.csv", index=False)
print(f"  files passing all gates       : {len(inv)}")
print(f"  unique (patient × tp) dropped : {len(drop_df)}  ({drop_df['reason'].value_counts().to_dict()})")
print(f"  drop log → {PROC/'mri_drop_log.csv'}")
pre = inv  # keep variable name used downstream


## 3. Pivot to one row per (patient × timepoint) — drop incomplete scans


In [ ]:
COMPLETE_MODS = ["brain_t1c","brain_t1n","brain_t2f","brain_t2w"]
wide = pre.pivot_table(index=["Patient_ID","Timepoint","Day_from_diag","y","Landmark_day","Death_day"],
                       columns="modality", values="path", aggfunc="first").reset_index()
wide.columns.name = None
keep_mask = wide[COMPLETE_MODS].notna().all(axis=1)
print(f"  scans before completeness filter: {len(wide)}")
print(f"  scans dropped (incomplete)      : {(~keep_mask).sum()}")
mri = wide[keep_mask].copy().reset_index(drop=True)
mri["Days_before_landmark"] = (mri["Landmark_day"] - mri["Day_from_diag"]).round().astype("Int64")
print(f"  final scans                     : {len(mri)}")
print(f"  unique eligible patients        : {mri['Patient_ID'].nunique()} / {len(elig)}")

# Final post-condition: zero post-landmark, zero post-death scans
assert (mri['Day_from_diag'] < mri['Landmark_day']).all(), "leak: post-landmark scan slipped through"
assert (mri['Death_day'].isna() | (mri['Day_from_diag'] <= mri['Death_day'])).all(), "leak: post-death scan slipped through"
print("  post-conditions OK: 0 post-landmark and 0 post-death scans in mri_paths.csv")


## 4. Save mri_paths.csv + summary


In [ ]:
mri.to_csv(PROC / "mri_paths.csv", index=False)
print(f"  wrote {PROC/'mri_paths.csv'}  ({len(mri)} rows × {mri.shape[1]} cols)")

per_pat = mri.groupby("Patient_ID").size().reindex(elig["Patient_ID"], fill_value=0)
summary = {
    "n_eligible":            int(len(elig)),
    "n_with_any_complete":   int((per_pat > 0).sum()),
    "n_without_any":         int((per_pat == 0).sum()),
    "median_scans":          float(per_pat.median()),
    "max_scans":             int(per_pat.max()),
    "modalities_required":   COMPLETE_MODS,
}
with open(PROC / "mri_summary.json", "w") as f: json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


## 5. Sanity preview


In [ ]:
mri.head(8)


## 6. Provision the VLM caption driver

We write `../VLM/run_radfm_captions.py` and `../VLM/README.md` so the
captioning job can be queued separately (it takes ≈ 2.5 h on an
A100 / M2-Max for the full cohort).


In [ ]:
VLM = Path("../VLM").resolve(); VLM.mkdir(exist_ok=True)

DRIVER = '''#!/usr/bin/env python
"""Caption every (patient × pre-T₂) scan with a medical VLM.

Tries the medical-VLM fallback ladder in order; the first model that
loads + runs successfully wins.  The model actually used is recorded
in `Dataset/Processed/mri_captions.meta.json` so the proposal write-up
matches reality.

Usage:
    python run_radfm_captions.py [--max N]   # caption first N scans (default = all)
"""
from __future__ import annotations
import argparse, json, sys, time, importlib
from pathlib import Path
import pandas as pd

ROOT      = Path(__file__).resolve().parent.parent
DATA_CSV  = ROOT / "Dataset/Processed/mri_paths.csv"
OUT_CSV   = ROOT / "Dataset/Processed/mri_captions.csv"
META_JSON = ROOT / "Dataset/Processed/mri_captions.meta.json"

LADDER = [
    ("RadFM",       "vlm_backends.radfm_backend",       "caption_volumes"),
    ("LLaVA-Med",   "vlm_backends.llava_med_backend",   "caption_volumes"),
    ("MAIRA-2",     "vlm_backends.maira2_backend",      "caption_volumes"),
    ("Med-Flamingo","vlm_backends.medflamingo_backend", "caption_volumes"),
]


def try_load(model_name, modpath, fnname):
    try:
        mod = importlib.import_module(modpath)
        fn  = getattr(mod, fnname)
        # Probe-load the model (this triggers downloads / weight init):
        _ = mod.load_model()
        return fn
    except Exception as e:
        print(f"  [{model_name}] unavailable: {type(e).__name__}: {e}")
        return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--max", type=int, default=0, help="0 = all rows")
    args = ap.parse_args()

    df = pd.read_csv(DATA_CSV)
    if args.max > 0:
        df = df.head(args.max)
    print(f"caption {len(df)} scans → {OUT_CSV}")

    caption_fn, model_used = None, None
    for name, modpath, fnname in LADDER:
        print(f"trying {name} ...")
        fn = try_load(name, modpath, fnname)
        if fn is not None:
            caption_fn, model_used = fn, name
            print(f"  ✓ using {name}")
            break
    if caption_fn is None:
        sys.exit("ERROR: no medical VLM in the fallback ladder loaded successfully. "
                 "Check pip installs and HF model availability.")

    t0 = time.time(); rows = []
    for i, r in df.iterrows():
        try:
            cap = caption_fn(t1c=r["brain_t1c"], t1n=r["brain_t1n"],
                             t2f=r["brain_t2f"], t2w=r["brain_t2w"])
        except Exception as e:
            cap = f"[CAPTION FAILED: {type(e).__name__}: {e}]"
        rows.append({"Patient_ID": r["Patient_ID"], "Timepoint": int(r["Timepoint"]),
                     "Day_from_diag": float(r["Day_from_diag"]),
                     "T2": float(r["T2"]), "y": int(r["y"]),
                     "caption": cap})
        if (i+1) % 5 == 0:
            print(f"  [{i+1}/{len(df)}]  elapsed {(time.time()-t0)/60:.1f} min", flush=True)
    pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
    META_JSON.write_text(json.dumps({
        "model_used":        model_used,
        "fallback_ladder":   [n for n, _, _ in LADDER],
        "n_scans_captioned": len(rows),
        "n_eligible_patients": int(df["Patient_ID"].nunique()),
        "elapsed_minutes":   round((time.time()-t0)/60, 2),
    }, indent=2))
    print(f"\nDONE — wrote {OUT_CSV}\n        and {META_JSON}")


if __name__ == "__main__":
    main()
'''
(VLM / "run_radfm_captions.py").write_text(DRIVER)
print(f"  wrote {VLM/'run_radfm_captions.py'}")


## 7. Stub the per-backend wrapper modules

Each wrapper exposes `load_model()` and `caption_volumes(t1c, t1n,
t2f, t2w) -> str`.  The actual model code is filled in when the VLM
captioning is queued (so notebook execution doesn't depend on
multi-GB downloads).


In [ ]:
BACKEND_DIR = (Path("../VLM") / "vlm_backends").resolve(); BACKEND_DIR.mkdir(exist_ok=True)
(BACKEND_DIR / "__init__.py").write_text("")

STUB = '''"""{name} backend stub.

Fill in `load_model()` and `caption_volumes(...)` when this rung of the
medical-VLM fallback ladder is selected.  Raise ImportError or
RuntimeError to skip to the next model in the ladder."""

def load_model():
    raise ImportError("{name} not yet wired in — falling through ladder")

def caption_volumes(t1c, t1n, t2f, t2w):
    raise NotImplementedError
'''
for fname, model in [
    ("radfm_backend.py",       "RadFM"),
    ("llava_med_backend.py",   "LLaVA-Med v1.5"),
    ("maira2_backend.py",      "MAIRA-2"),
    ("medflamingo_backend.py", "Med-Flamingo"),
]:
    p = BACKEND_DIR / fname
    if not p.exists():
        p.write_text(STUB.format(name=model))
        print(f"  wrote stub {p}")
    else:
        print(f"  keep existing {p}")


In [ ]:
README = '''# VLM caption pipeline (Exp iv — Second Progression)

## What this folder produces

`Dataset/Processed/mri_captions.csv` — one caption per (patient × pre-T₂
MRI timepoint), used by the Mistral RAG prompt builder for Exp(iv).

## How it picks a model

`run_radfm_captions.py` walks the medical-VLM fallback ladder in
priority order:

1. **RadFM** (Wu et al., 2024) — primary; 13B; multi-volume native.
2. **LLaVA-Med v1.5** (Microsoft, 2024) — 7B; PMC-15M biomedical.
3. **MAIRA-2** (Microsoft, 2024) — 7B; radiology report generation.
4. **Med-Flamingo** (Stanford, 2023) — 9B; few-shot medical multi-modal.

The first model that successfully imports + initialises wins.  The
choice is recorded in `Dataset/Processed/mri_captions.meta.json`.

## Run it

```bash
conda activate beep-env
cd Second_Recur/VLM
python run_radfm_captions.py            # full cohort, ~2.5h on A100/M2-Max
python run_radfm_captions.py --max 10   # smoke test (first 10 scans)
```

## Wiring a backend

Each `vlm_backends/*.py` is a stub.  When a model is chosen, fill in:

```python
def load_model():
    # download / instantiate the model (cache it module-globally)
    ...

def caption_volumes(t1c, t1n, t2f, t2w):
    # paths to the 4 NIfTI files for one (patient × timepoint)
    return "string caption"
```

## Citations (for the report)

- **RadFM**: Wu, C. *et al.* "Towards Generalist Foundation Model for
  Radiology" — arXiv:2308.02463 (2024).
- **LLaVA-Med**: Li, C. *et al.* "LLaVA-Med: Training a Large
  Language-and-Vision Assistant for Biomedicine in One Day" — NeurIPS 2023.
- **MAIRA-2**: Bannur, S. *et al.* "MAIRA-2: Grounded Radiology
  Report Generation" — arXiv:2406.04449 (2024).
- **Med-Flamingo**: Moor, M. *et al.* "Med-Flamingo: a Multimodal
  Medical Few-shot Learner" — Proceedings of ML4H 2023.
'''
(VLM / "README.md").write_text(README)
print(f"  wrote {VLM/'README.md'}")


## 8. Done

Now `Image_preprocessing.ipynb` has produced everything Exp(iv)
needs.  Caption generation is queued via `VLM/run_radfm_captions.py`
and runs separately from the model-training loop.
